<a href="https://colab.research.google.com/github/Ridhi0812/Perceptra/blob/main/Perceptra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#updated cnn code ,

import os
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Device Selection
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

class conv_net(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # Block 1: 64x64 → 32x32
        self.conv1_1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv1_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Block 2: 32x32 → 16x16
        self.conv2_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv2_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Block 3: 16x16 → 8x8
        self.conv3_1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv3_2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv3_3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Block 4: 8x8 → 4x4
        self.conv4_1 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.conv4_2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv4_3 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Block 5: 4x4 → 2x2
        self.conv5_1 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_3 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.pool5 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully connected layers
        self.fc1 = nn.Linear(512 * 2 * 2, 4096)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(4096, 512)
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(512,256)
        self.dropout3 = nn.Dropout(0.5)
        self.fc4 = nn.Linear(256, num_classes)

    def forward(self, x):
        # Block 1
        x = F.relu(self.conv1_1(x))
        x = F.relu(self.conv1_2(x))
        x = self.pool1(x)

        # Block 2
        x = F.relu(self.conv2_1(x))
        x = F.relu(self.conv2_2(x))
        x = self.pool2(x)

        # Block 3
        x = F.relu(self.conv3_1(x))
        x = F.relu(self.conv3_2(x))
        x = F.relu(self.conv3_3(x))
        x = self.pool3(x)

        # Block 4
        x = F.relu(self.conv4_1(x))
        x = F.relu(self.conv4_2(x))
        x = F.relu(self.conv4_3(x))
        x = self.pool4(x)

        # Block 5
        x = F.relu(self.conv5_1(x))
        x = F.relu(self.conv5_2(x))
        x = F.relu(self.conv5_3(x))
        x = self.pool5(x)

        # Flatten
        x = x.view(x.size(0), -1)

        # Fully connected layers
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)

        dim_output = x  # 256-dim feature vector
        x = F.relu(self.fc3(x))
        x = self.dropout3(x)
        x = self.fc4(x)

        return x, dim_output

def image_format(path_to_image):
    img = cv2.imread(path_to_image)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (64, 64))

    img_tensor = torch.tensor(img, dtype=torch.float32)
    img_tensor = img_tensor.permute(2, 0, 1) / 255.0
    img_tensor = img_tensor.unsqueeze(0)

    return img_tensor


def data_input(root_folder, batch_size):
    class_mapping = {cls: i for i, cls in enumerate(os.listdir(root_folder))}
    img_batches = []

    imgs, labs = [], []

    for cls in class_mapping:
        folder = os.path.join(root_folder, cls)
        for file_name in os.listdir(folder):
            path = os.path.join(folder, file_name)

            tensor = image_format(path)

            imgs.append(tensor)
            labs.append(class_mapping[cls])

            if len(imgs) == batch_size:
                batch_tensor = torch.cat(imgs, dim=0)
                batch_labels = torch.tensor(labs, dtype=torch.long)

                # Move Tensors to Device
                batch_tensor = batch_tensor.to(DEVICE)
                batch_labels = batch_labels.to(DEVICE)

                img_batches.append((batch_tensor, batch_labels))

                imgs, labs = [], []

    # Handle the remaining partial batch
    if len(imgs) > 0:
        batch_tensor = torch.cat(imgs, dim=0)
        batch_labels = torch.tensor(labs, dtype=torch.long)

        batch_tensor = batch_tensor.to(DEVICE)
        batch_labels = batch_labels.to(DEVICE)

        img_batches.append((batch_tensor, batch_labels))

    return img_batches, len(class_mapping)


def train_model(img_batches, num_classes , epochs):
    model = conv_net(num_classes)
    # Move Model to Device
    model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

    for epoch in range(epochs):
        print(f"training for epoch : {epoch}")

        model.train() # Set model to training mode
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        for batch_img, batch_lbl in img_batches:
            # Data is already on DEVICE

            optimizer.zero_grad()
            logits, _ = model(batch_img)
            loss = criterion(logits, batch_lbl)
            loss.backward()
            optimizer.step()

            # --- Loss and Accuracy Calculation ---
            running_loss += loss.item() * batch_img.size(0)

            # Get the predicted class index
            _, predicted = torch.max(logits.data, 1)

            # Count total samples and correct predictions
            total_samples += batch_lbl.size(0)
            correct_predictions += (predicted == batch_lbl).sum().item()

        # Calculate average loss and accuracy for the epoch
        avg_loss = running_loss / total_samples
        accuracy = 100 * correct_predictions / total_samples

        print(f"epoch {epoch} | loss = {avg_loss:.4f} | accuracy = {accuracy:.2f}%")
        torch.save(model.state_dict(), f"checkpoint_{epoch}.pth")


def inference(model_path, num_classes, img_path):
    model = conv_net(num_classes)
    # Move Model to Device
    model.to(DEVICE)

    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()

    img_tensor = image_format(img_path)
    # Move Input Tensor to Device
    img_tensor = img_tensor.to(DEVICE)

    with torch.no_grad():
        output, dim = model(img_tensor)

    # Move output back to CPU for item retrieval
    predicted_class = torch.argmax(output.cpu()).item()

    return predicted_class


def main():
    print("welcome to model trainer")
    root_folder = input("enter root folder path : ")
    batch_size  = int(input("enter batch size : "))
    epoch       = int(input("enter number of epochs : "))

    tensor_data, num_classes = data_input(root_folder, batch_size)

    print("training has started")
    train_model(tensor_data, num_classes, epoch)


if __name__ == "__main__":
    main()

Using device: cuda
welcome to model trainer
enter root folder path : /content/drive/MyDrive/Animals_Photo
enter batch size : 10
enter number of epochs : 200
training has started
training for epoch : 0
epoch 0 | loss = 10.9758 | accuracy = 18.40%
training for epoch : 1
epoch 1 | loss = 1.9184 | accuracy = 20.70%
training for epoch : 2
epoch 2 | loss = 1.7138 | accuracy = 25.40%
training for epoch : 3
epoch 3 | loss = 1.6483 | accuracy = 25.50%
training for epoch : 4
epoch 4 | loss = 1.6815 | accuracy = 21.10%
training for epoch : 5
epoch 5 | loss = 1.6248 | accuracy = 16.10%
training for epoch : 6
epoch 6 | loss = 4.0949 | accuracy = 35.80%
training for epoch : 7
epoch 7 | loss = 1.4555 | accuracy = 38.00%
training for epoch : 8
epoch 8 | loss = 1.6435 | accuracy = 20.90%
training for epoch : 9
epoch 9 | loss = 1.6480 | accuracy = 20.60%
training for epoch : 10
epoch 10 | loss = 1.6907 | accuracy = 13.90%
training for epoch : 11
epoch 11 | loss = 1.6500 | accuracy = 8.20%
training for e

In [ ]:
# image_text_model_512.py

import os
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from transformers import T5ForConditionalGeneration, T5Tokenizer
from torch.utils.data import Dataset, DataLoader

# ==================== Device ====================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ==================== Modified CNN Model - 512-dim output ====================
class conv_net(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # Block 1: 64x64 → 32x32
        self.conv1_1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv1_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Block 2: 32x32 → 16x16
        self.conv2_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv2_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Block 3: 16x16 → 8x8
        self.conv3_1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv3_2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv3_3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Block 4: 8x8 → 4x4
        self.conv4_1 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.conv4_2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv4_3 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Block 5: 4x4 → 2x2
        self.conv5_1 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_3 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.pool5 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully connected layers
        self.fc1 = nn.Linear(512 * 2 * 2, 4096)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(4096, 512)
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(512,256)
        self.dropout3 = nn.Dropout(0.5)
        self.fc4 = nn.Linear(256, num_classes)

    def forward(self, x):
        # Block 1
        x = F.relu(self.conv1_1(x))
        x = F.relu(self.conv1_2(x))
        x = self.pool1(x)

        # Block 2
        x = F.relu(self.conv2_1(x))
        x = F.relu(self.conv2_2(x))
        x = self.pool2(x)

        # Block 3
        x = F.relu(self.conv3_1(x))
        x = F.relu(self.conv3_2(x))
        x = F.relu(self.conv3_3(x))
        x = self.pool3(x)

        # Block 4
        x = F.relu(self.conv4_1(x))
        x = F.relu(self.conv4_2(x))
        x = F.relu(self.conv4_3(x))
        x = self.pool4(x)

        # Block 5
        x = F.relu(self.conv5_1(x))
        x = F.relu(self.conv5_2(x))
        x = F.relu(self.conv5_3(x))
        x = self.pool5(x)

        # Flatten
        x = x.view(x.size(0), -1)

        # Fully connected layers
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)

        dim_output = x  # 256-dim feature vector
        x = F.relu(self.fc3(x))
        x = self.dropout3(x)
        x = self.fc4(x)

        return x, dim_output

# ==================== Simplified Image-Text Model ====================
class ImageTextModel(nn.Module):
    def __init__(self, cnn_weights_path, num_classes, t5_model_name='t5-small'):
        super().__init__()
        # Load pre-trained CNN
        self.image_encoder = conv_net(num_classes)
        if cnn_weights_path and os.path.exists(cnn_weights_path):
            self.image_encoder.load_state_dict(torch.load(cnn_weights_path, map_location=DEVICE))
            print(f"Loaded CNN weights from {cnn_weights_path}")

        # Freeze CNN weights (optional)
        for param in self.image_encoder.parameters():
            param.requires_grad = False

        # T5 model
        self.t5_model = T5ForConditionalGeneration.from_pretrained(t5_model_name)
        self.tokenizer = T5Tokenizer.from_pretrained(t5_model_name)

        # NO PROJECTION LAYER NEEDED! 512-dim directly matches T5-small's embedding dimension

    def forward(self, image, input_text, target_text=None):
        # Get 512-dim image features DIRECTLY
        _, img_features = self.image_encoder(image)  # [batch, 512]

        # Just unsqueeze - no projection needed!
        img_embedded = img_features.unsqueeze(1)  # [batch, 1, 512]

        # Tokenize input text
        if isinstance(input_text, list):
            input_ids = self.tokenizer(input_text, return_tensors="pt", padding=True, truncation=True).input_ids.to(image.device)
        else:
            input_ids = input_text.to(image.device)

        # Get T5 text embeddings
        text_embedded = self.t5_model.encoder.embed_tokens(input_ids)  # [batch, seq_len, 512]

        # Concatenate image embedding at the beginning
        combined = torch.cat([img_embedded, text_embedded], dim=1)  # [batch, 1+seq_len, 512]

        # Attention mask
        attention_mask = torch.ones(combined.size()[:-1], dtype=torch.long).to(image.device)

        # Encode combined features
        encoder_outputs = self.t5_model.encoder(
            inputs_embeds=combined,
            attention_mask=attention_mask,
            return_dict=True
        )

        if target_text is not None:
            # Training mode
            if isinstance(target_text, list):
                target_ids = self.tokenizer(target_text, return_tensors="pt", padding=True, truncation=True).input_ids.to(image.device)
            else:
                target_ids = target_text.to(image.device)

            target_ids[target_ids == self.tokenizer.pad_token_id] = -100
            outputs = self.t5_model(
                encoder_outputs=encoder_outputs,
                attention_mask=attention_mask,
                labels=target_ids
            )
            return outputs.loss
        else:
            # Inference mode
            outputs = self.t5_model.generate(
                encoder_outputs=encoder_outputs,
                attention_mask=attention_mask,
                max_length=100,
                num_beams=4,
                early_stopping=True
            )
            return outputs

# ==================== Dataset ====================
class ImageTextDataset(Dataset):
    def __init__(self, excel_path, image_column='image_path', label_column = "animal_class",  text_column='text'):
        self.df = pd.read_excel(excel_path)
        self.image_column = image_column
        self.text_column = text_column
        self.label_column = label_column

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        root = "/content/drive/MyDrive/Animals_Photo/"
        row = self.df.iloc[idx]
        label = row[self.label_column]

        image_path = root + label + "/" + row[self.image_column]
        text = row[self.text_column]

        # Load and preprocess image
        try :
          img = cv2.imread(image_path)
          img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
          img = cv2.resize(img, (64, 64))
          img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0
          return img_tensor, text
        except :
            print("file not found")


def collate_fn(batch):
    images = torch.stack([item[0] for item in batch])
    texts = [item[1] for item in batch]
    return images, texts

# ==================== Training ====================
def train_model(excel_path, cnn_weights_path, num_classes, epochs=10, batch_size=8, learning_rate=1e-4):
    dataset = ImageTextDataset(excel_path)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

    model = ImageTextModel(cnn_weights_path, num_classes).to(DEVICE)

    # Only T5 parameters need training now (no projection layer!)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        num_batches = 0

        for batch_idx, (images, texts) in enumerate(dataloader):
            images = images.to(DEVICE)
            input_prompts = ["describe image: " for _ in texts]
            loss = model(images, input_prompts, texts)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

            if (batch_idx + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Batch [{batch_idx+1}/{len(dataloader)}], Loss: {loss.item():.4f}")

        avg_loss = total_loss / num_batches
        print(f"Epoch [{epoch+1}/{epochs}] - Average Loss: {avg_loss:.4f}")

        torch.save(model.state_dict(), f"image_text_model_epoch_{epoch+1}.pth")
        print(f"Saved checkpoint: image_text_model_epoch_{epoch+1}.pth")

    return model

# ==================== Inference ====================
def extract_features_and_generate_text(image_path, model_path, cnn_weights_path, num_classes):
    model = ImageTextModel(cnn_weights_path, num_classes).to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()

    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (64, 64))
    img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0
    img_tensor = img_tensor.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        _, features = model.image_encoder(img_tensor)
        output_ids = model(img_tensor, ["describe image: "])
        generated_text = model.tokenizer.decode(output_ids[0], skip_special_tokens=True)

    return features.cpu().numpy()[0], generated_text

def batch_inference(excel_path, model_path, cnn_weights_path, num_classes, output_excel_path):
    df = pd.read_excel(excel_path)
    model = ImageTextModel(cnn_weights_path, num_classes).to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()

    features_list = []
    generated_texts = []

    for idx, row in df.iterrows():
        image_path = row['image_path']
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (64, 64))
        img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0
        img_tensor = img_tensor.unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            _, features = model.image_encoder(img_tensor)
            output_ids = model(img_tensor, ["describe image: "])
            generated_text = model.tokenizer.decode(output_ids[0], skip_special_tokens=True)

        features_list.append(features.cpu().numpy()[0])
        generated_texts.append(generated_text)
        print(f"Processed {idx+1}/{len(df)}: {image_path}")

    df['generated_text'] = generated_texts
    df['features_512d'] = [str(f.tolist()) for f in features_list]
    df.to_excel(output_excel_path, index=False)
    print(f"Results saved to {output_excel_path}")

# ==================== Main ====================
if __name__ == "__main__":
    print("=" * 50)
    print("Image-Text Model with CNN and T5 (512-dim)")
    print("=" * 50)
    print("\n1. Train model")
    print("2. Run inference on single image")
    print("3. Run batch inference from Excel")
    choice = input("\nEnter your choice (1/2/3): ")

    if choice == "1":
        excel_path = input("Enter path to Excel file (with 'image_path' and 'text' columns): ")
        cnn_weights_path = input("Enter path to CNN weights (.pth file): ")
        num_classes = int(input("Enter number of classes in CNN: "))
        epochs = int(input("Enter number of epochs (default 10): ") or "10")
        batch_size = int(input("Enter batch size (default 8): ") or "8")
        print("\nStarting training...")
        train_model(excel_path, cnn_weights_path, num_classes, epochs, batch_size)
        print("\nTraining completed!")

    elif choice == "2":
        image_path = input("Enter path to image: ")
        model_path = input("Enter path to trained model: ")
        cnn_weights_path = input("Enter path to CNN weights: ")
        num_classes = int(input("Enter number of classes in CNN: "))
        features, text = extract_features_and_generate_text(image_path, model_path, cnn_weights_path, num_classes)
        print("\n" + "=" * 50)
        print("Results:")
        print("=" * 50)
        print(f"\n512-dim features shape: {features.shape}")
        print(f"Generated text: {text}")

    elif choice == "3":
        excel_path = input("Enter path to input Excel file: ")
        model_path = input("Enter path to trained model: ")
        cnn_weights_path = input("Enter path to CNN weights: ")
        num_classes = int(input("Enter number of classes in CNN: "))
        output_excel_path = input("Enter path to save output Excel: ")
        print("\nRunning batch inference...")
        batch_inference(excel_path, model_path, cnn_weights_path, num_classes, output_excel_path)
        print("\nBatch inference completed!")

    else:
        print("Invalid choice!")

Using device: cpu
Image-Text Model with CNN and T5 (512-dim)

1. Train model
2. Run inference on single image
3. Run batch inference from Excel

Enter your choice (1/2/3): 2
Enter path to image: /content/drive/MyDrive/Animals_Photo/cat/ea36b70629f5073ed1584d05fb1d4e9fe777ead218ac104497f5c978a7e8b7bc_640.jpg
Enter path to trained model: /content/drive/MyDrive/Animals_Photo/image_text_model_epoch_60.pth
Enter path to CNN weights: /content/drive/MyDrive/Animals_Photo/checkpoint_110.pth
Enter number of classes in CNN: 5
Loaded CNN weights from /content/drive/MyDrive/Animals_Photo/checkpoint_110.pth

Results:

512-dim features shape: (512,)
Generated text: Cat open his mouth


In [1]:
!git config --global user.name "Ridhi0812"
!git config --global user.email "ridhisambhor08@gmail.com"

In [2]:
!git clone https://github.com/Ridhi0812/Perceptra.git

Cloning into 'Perceptra'...


In [3]:
!cp -r /content/project/* /content/Perceptra/

cp: cannot stat '/content/project/*': No such file or directory


In [4]:
!ls /content

Perceptra  sample_data


In [5]:
!ls /content
!ls /content/Perceptra

Perceptra  sample_data


In [6]:
!ls /content/Perceptra

In [7]:
!pwd

/content


In [8]:
!ls -la /content/Perceptra

total 12
drwxr-xr-x 3 root root 4096 Sep  4 16:41 .
drwxr-xr-x 1 root root 4096 Sep  4 16:41 ..
drwxr-xr-x 7 root root 4096 Sep  4 16:41 .git
